# Étude de cas : ventes et livraisons

Une entreprise vend des produits en ligne (épicerie, frais, boissons, hygiène, maison, électronique, livres, bricolage, collection) et les livre elle-même, depuis ses entrepôts de la région parisienne, avec sa propre flotte de véhicules.

### Les fichiers (`/work/data`)

| Fichier | Contenu |
|---|---|
| `produits.csv` | le catalogue : `produit_id, nom, categorie, sous_categorie, prix, poids_kg` |
| `commandes.csv` | le journal des commandes, une ligne par événement : `commande_id, client_id, horodatage, evenement, livreur_id, detail` |
| `vehicules/<vehicule>/<AAAA-MM>.csv` | la trace GPS de chaque véhicule, **un point par minute** : `timestamp, livreur_id, lon, lat, vitesse_kmh, dans_vehicule, charge_kg` |
| `communes.csv` | les communes de Paris et de la petite couronne et les 20 arrondissements de Paris, avec leur centre : `commune, code_postal, lon, lat` |
| `communes.geojson` | les contours des mêmes communes (propriétés `commune`, `code_postal`) |

### Le journal des commandes

Une commande apparaît plusieurs fois : une ligne par étape.

| `evenement` | Sens | `livreur_id` | `detail` |
|---|---|---|---|
| `commande` | le client passe commande | — | les produits : `P0201x1\|P0033x4` (produit `x` quantité) |
| `chargee` | la commande est chargée dans un véhicule | le livreur | — |
| `livree` | le client signe ; horodatage à la seconde | le livreur | — |
| `echec` | la livraison n'a pas pu se faire | le livreur | le motif |
| `reportee` | chargée mais pas tentée : rapportée à l'entrepôt en fin de service | le livreur | — |
| `retour` | la commande est abandonnée et retournée à l'entrepôt | le livreur | — |

### Les traces GPS

Un livreur conduit un seul véhicule par jour. `dans_vehicule` vaut `false` quand le livreur est descendu du véhicule ; `charge_kg` est le poids transporté.

### Les services

| Adresse | Service |
|---|---|
| http://localhost:8888 | JupyterLab |
| http://localhost:8080 | Master Spark |
| http://localhost:4040 | Spark UI de la session ouverte |
| http://localhost:18080 | Spark History |
| `mongodb://mongo:27017` | MongoDB |

## 1) Charger la donnée

1.1) La session Spark et les fonctions fournies.

In [7]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import types as T
from pyspark.sql.window import Window

spark = (SparkSession.builder
         .appName("livraisons")
         .master("spark://spark-master:7077")
         .getOrCreate())
DATA = "/work/data"
RESULTATS = "/work/resultats"


def haversine(lon1, lat1, lon2, lat2):
    """Distance en km entre deux points : noms de colonnes ou colonnes, en degrés."""
    lon1, lat1, lon2, lat2 = [F.col(c) if isinstance(c, str) else c for c in (lon1, lat1, lon2, lat2)]
    dlon = F.radians(lon2 - lon1)
    dlat = F.radians(lat2 - lat1)
    a = F.sin(dlat / 2) ** 2 + F.cos(F.radians(lat1)) * F.cos(F.radians(lat2)) * F.sin(dlon / 2) ** 2
    return 2 * 6371.0 * F.asin(F.sqrt(a))


spark.conf.set("spark.sql.session.timeZone", "UTC")

1.2) Définissez un schéma explicite (`StructType`) pour chacun des 4 fichiers CSV, puis chargez `produits.csv`, `commandes.csv`, `communes.csv` et **tous** les fichiers GPS en une seule lecture (`vehicules/*/*.csv`). Les horodatages doivent être de type `timestamp`, `dans_vehicule` de type `boolean`, les nombres de type `double`.

Affichez le nombre de lignes de chaque DataFrame.



In [8]:
schema_produits = (T.StructType()
                   .add("produit_id", T.StringType()).add("nom", T.StringType())
                   .add("categorie", T.StringType()).add("sous_categorie", T.StringType())
                   .add("prix", T.DoubleType()).add("poids_kg", T.DoubleType()))

schema_commandes = (T.StructType()
                    .add("commande_id", T.StringType()).add("client_id", T.StringType())
                    .add("horodatage", T.TimestampType()).add("evenement", T.StringType())
                    .add("livreur_id", T.StringType()).add("detail", T.StringType()))

schema_communes = (T.StructType()
                   .add("commune", T.StringType()).add("code_postal", T.StringType())
                   .add("lon", T.DoubleType()).add("lat", T.DoubleType()))

schema_gps = (T.StructType()
              .add("timestamp", T.TimestampType()).add("livreur_id", T.StringType())
              .add("lon", T.DoubleType()).add("lat", T.DoubleType())
              .add("vitesse_kmh", T.DoubleType()).add("dans_vehicule", T.BooleanType())
              .add("charge_kg", T.DoubleType()))


def lire(chemin, schema):
    return spark.read.csv(chemin, header=True, schema=schema, timestampFormat="yyyy-MM-dd HH:mm:ss")


df_produits = lire(f"{DATA}/produits.csv", schema_produits).cache()
df_communes = lire(f"{DATA}/communes.csv", schema_communes).cache()
df_commandes = lire(f"{DATA}/commandes.csv", schema_commandes).repartition("commande_id").cache()
df_gps = lire(f"{DATA}/vehicules/*/*.csv", schema_gps)      


def nb(condition):
    """Nombre de lignes vérifiant la condition (à utiliser dans agg)."""
    return F.sum(condition.cast("int"))


for nom, df in [("produits", df_produits), ("commandes", df_commandes), ("communes", df_communes), ("gps", df_gps)]:
    print(nom, df.count())

produits 233


commandes 846535
communes 142


gps 5174905


1.3) L'identifiant du véhicule n'est pas une colonne des fichiers GPS : il est dans le chemin du fichier. Ajoutez une colonne `vehicule_id` (fonction `F.input_file_name()`), et une colonne `jour` (la date du point).

In [9]:
gps = (df_gps
       .withColumn("vehicule_id", F.regexp_extract(F.input_file_name(), r"/vehicules/([^/]+)/", 1))
       .withColumn("jour", F.to_date("timestamp"))
       .repartition("livreur_id", "jour")
       .sortWithinPartitions("livreur_id", "jour", "timestamp")
       .cache())
gps_base = gps
print(gps.count())
gps.show(5)

5174905
+-------------------+----------+--------+---------+-----------+-------------+---------+-----------+----------+
|          timestamp|livreur_id|     lon|      lat|vitesse_kmh|dans_vehicule|charge_kg|vehicule_id|      jour|
+-------------------+----------+--------+---------+-----------+-------------+---------+-----------+----------+
|2025-05-09 07:41:00|      L002|2.277057|48.936735|        0.0|        false|   1092.3|        V03|2025-05-09|
|2025-05-09 07:42:00|      L002|2.277058|48.936734|        0.0|        false|   1092.3|        V03|2025-05-09|
|2025-05-09 07:43:00|      L002|2.277057|48.936735|        0.0|        false|   1092.3|        V03|2025-05-09|
|2025-05-09 07:44:00|      L002|2.277056|48.936735|        0.0|        false|   1092.3|        V03|2025-05-09|
|2025-05-09 07:45:00|      L002|2.277056|48.936735|        0.0|        false|   1092.3|        V03|2025-05-09|
+-------------------+----------+--------+---------+-----------+-------------+---------+-----------+-----

## 2) Vérifier la donnée

2.1) Combien y a-t-il de commandes, de clients, de livreurs et de véhicules ?

In [10]:
df_commandes.agg(F.countDistinct("commande_id").alias("commandes"),
                 F.countDistinct("client_id").alias("clients")).show()
gps.agg(F.countDistinct("livreur_id").alias("livreurs"),
        F.countDistinct("vehicule_id").alias("vehicules")).show()

+---------+-------+
|commandes|clients|
+---------+-------+
|   271617|   8000|
+---------+-------+



+--------+---------+
|livreurs|vehicules|
+--------+---------+
|      90|       72|
+--------+---------+



2.2) Comptez les événements du journal par type.

In [11]:
df_evenements_compte = df_commandes.groupBy("evenement").count()
df_evenements_compte.show()

+---------+------+
|evenement| count|
+---------+------+
| commande|271617|
|    echec| 11220|
| reportee|  6960|
|   livree|268682|
|  chargee|286862|
|   retour|  1194|
+---------+------+



2.3) Pour chaque commande, comptez ses événements `chargee`. Combien de commandes ont été chargées 0, 1, 2, 3… fois ?

In [12]:
nb_chargees = df_commandes.groupBy("commande_id").agg(nb(F.col("evenement") == "chargee").alias("nb_chargee"))
nb_chargees.groupBy("nb_chargee").count().orderBy("nb_chargee").show()

+----------+------+
|nb_chargee| count|
+----------+------+
|         0|  1637|
|         1|254229|
|         2| 14673|
|         3|  1026|
|         4|    51|
|         5|     1|
+----------+------+



2.4) Pour chaque événement `chargee`, trouvez l'événement qui le précède dans la vie de la même commande. Comptez les `chargee` par événement précédent.

In [13]:
w_commande = Window.partitionBy("commande_id").orderBy("horodatage")

(df_commandes.withColumn("precedent", F.lag("evenement").over(w_commande))
 .where(F.col("evenement") == "chargee")
 .groupBy("precedent").count().show())

+---------+------+
|precedent| count|
+---------+------+
| commande|269980|
|    echec|  9950|
| reportee|  6932|
+---------+------+



*Un rechargement suit toujours un `echec` (nouvelle tentative) ou un `reportee` (commande rapportée en fin de service) : il y a plus de `chargee` que de `commande`.*

2.5) Au 31/12/2025, combien de commandes ne sont pas livrées ? Séparez les commandes abandonnées (événement `retour`) des commandes encore en cours.

In [14]:
statut = (df_commandes.where(F.col("horodatage") < "2026-01-01")
          .groupBy("commande_id")
          .agg(F.max(F.col("evenement") == "livree").alias("livree"),
               F.max(F.col("evenement") == "retour").alias("retour")))

(statut.select(F.when(F.col("livree"), "livrée")
                .when(F.col("retour"), "abandonnée (retour)")
                .otherwise("en cours").alias("statut"))
 .groupBy("statut").count().show())

+-------------------+------+
|             statut| count|
+-------------------+------+
|           en cours|  1741|
|abandonnée (retour)|  1194|
|             livrée|268682|
+-------------------+------+



## 3) Les lignes de commande

3.1) Les produits d'une commande sont dans la colonne `detail` de l'événement `commande`. Construisez la table `lignes` : une ligne par produit commandé, avec `commande_id`, `client_id`, `horodatage`, `produit_id`, `quantite`.

In [15]:
motif = r"^(P\d+)x(\d+)$"      # ex. « P0201x1 »

lignes = (df_commandes.where(F.col("evenement") == "commande")
          .select("commande_id", "client_id", "horodatage",
                  F.explode(F.split("detail", r"\|")).alias("item"))
          .select("commande_id", "client_id", "horodatage",
                  F.regexp_extract("item", motif, 1).alias("produit_id"),
                  F.regexp_extract("item", motif, 2).cast("int").alias("quantite")))
lignes.show(5)

+-----------+---------+-------------------+----------+--------+
|commande_id|client_id|         horodatage|produit_id|quantite|
+-----------+---------+-------------------+----------+--------+
|   C0000399|  CL06992|2025-01-01 21:04:19|     P0057|       2|
|   C0000399|  CL06992|2025-01-01 21:04:19|     P0083|       1|
|   C0000399|  CL06992|2025-01-01 21:04:19|     P0039|       3|
|   C0000399|  CL06992|2025-01-01 21:04:19|     P0066|       3|
|   C0000399|  CL06992|2025-01-01 21:04:19|     P0004|       4|
+-----------+---------+-------------------+----------+--------+
only showing top 5 rows



3.2) Ajoutez à `lignes` les colonnes du catalogue et le `montant` de chaque ligne. Construisez la table `commandes` : une ligne par commande, avec `montant`, `poids_kg` et `nb_lignes`.

In [16]:
lignes = (lignes.join(F.broadcast(df_produits), "produit_id")
          .withColumn("montant", F.round(F.col("prix") * F.col("quantite"), 2))
          .cache())

commandes = (lignes.groupBy("commande_id", "client_id", "horodatage")
             .agg(F.round(F.sum("montant"), 2).alias("montant"),
                  F.round(F.sum(F.col("poids_kg") * F.col("quantite")), 3).alias("poids_kg"),
                  F.count("*").alias("nb_lignes"))
             .cache())
commandes.show(5)

+-----------+---------+-------------------+-------+--------+---------+
|commande_id|client_id|         horodatage|montant|poids_kg|nb_lignes|
+-----------+---------+-------------------+-------+--------+---------+
|   C0000399|  CL06992|2025-01-01 21:04:19|   52.7|  19.425|        9|
|   C0000568|  CL06743|2025-01-02 08:48:06|  207.7|   55.72|       28|
|   C0000631|  CL04375|2025-01-02 11:15:20|  290.9|   297.3|       11|
|   C0000701|  CL00914|2025-01-02 14:23:43|  474.5|   485.6|        7|
|   C0001377|  CL00031|2025-01-03 20:24:27|  194.3|   57.87|       22|
+-----------+---------+-------------------+-------+--------+---------+
only showing top 5 rows



## 4) Les entrepôts

4.1) Pour chaque véhicule et chaque jour, gardez le premier et le dernier point GPS. Arrondissez leurs coordonnées à 0,01° et comptez les journées par position arrondie. Combien d'entrepôts y a-t-il ?

In [17]:
bornes = (gps.groupBy("vehicule_id", "jour")
          .agg(F.min(F.struct("timestamp", "lon", "lat")).alias("premier"),
               F.max(F.struct("timestamp", "lon", "lat")).alias("dernier"))
          .cache())


def points(champ):
    return bornes.select("vehicule_id", "jour",
                         F.col(champ + ".lon").alias("lon"), F.col(champ + ".lat").alias("lat"),
                         F.round(champ + ".lon", 2).alias("lon_r"), F.round(champ + ".lat", 2).alias("lat_r"))


premiers, derniers = points("premier"), points("dernier")

positions = (premiers.union(derniers)
             .select("vehicule_id", "jour", "lon_r", "lat_r").distinct()    # une journée compte une fois par position
             .groupBy("lon_r", "lat_r").agg(F.count("*").alias("nb_journees"))
             .orderBy(F.desc("nb_journees")).cache())
positions.show(20)

positions_entrepots = positions.where(F.col("nb_journees") >= 0.01 * bornes.count())
print("Nombre d'entrepôts :", positions_entrepots.count())

+-----+-----+-----------+
|lon_r|lat_r|nb_journees|
+-----+-----+-----------+
| 2.45|48.91|       4217|
| 2.28|48.94|       3208|
| 2.35|48.75|       2967|
+-----+-----+-----------+



Nombre d'entrepôts : 3


4.2) Construisez la table `entrepots` (`entrepot`, `lon`, `lat`) : `E1`, `E2`… par longitude croissante, position = moyenne des premiers points. Construisez la table `vehicule_entrepot` (`vehicule_id`, `entrepot`) et comptez les véhicules par entrepôt.

In [18]:
moy = (premiers.join(F.broadcast(positions_entrepots), ["lon_r", "lat_r"])
       .groupBy("lon_r", "lat_r").agg(F.avg("lon").alias("lon"), F.avg("lat").alias("lat"))
       .toPandas().sort_values("lon").reset_index(drop=True))
moy["entrepot"] = ["E%d" % (i + 1) for i in range(len(moy))]

entrepots_cle = spark.createDataFrame(moy)                   
entrepots = entrepots_cle.select("entrepot", "lon", "lat")
entrepots.show(truncate=False)
ENTREPOTS = [(r["entrepot"], r["lon"], r["lat"]) for r in entrepots.collect()]

vehicule_entrepot = (premiers.join(F.broadcast(entrepots_cle.select("entrepot", "lon_r", "lat_r")), ["lon_r", "lat_r"])
                     .groupBy("vehicule_id", "entrepot").count()
                     .groupBy("vehicule_id").agg(F.max(F.struct("count", "entrepot")).alias("m"))
                     .select("vehicule_id", F.col("m.entrepot").alias("entrepot"))
                     .cache())
vehicule_entrepot.groupBy("entrepot").count().orderBy("entrepot").show()

+--------+------------------+------------------+
|entrepot|lon               |lat               |
+--------+------------------+------------------+
|E1      |2.2770689133416475|48.93671290461345 |
|E2      |2.3515135942028986|48.74704698584426 |
|E3      |2.4489591059995255|48.906701645482585|
+--------+------------------+------------------+



+--------+-----+
|entrepot|count|
+--------+-----+
|      E1|   24|
|      E2|   24|
|      E3|   24|
+--------+-----+



## 5) Les livraisons

On découpe la journée de chaque livreur en **livraisons** : une livraison commence quand le livreur descend du véhicule et dure jusqu'à la livraison suivante. Le chargement du matin, les retours à l'entrepôt et la pause déjeuner sont aussi des périodes à pied : la question 5.6 les distingue.

5.1) Ajoutez à `gps` une colonne `changement` : 1 si le livreur vient de descendre du véhicule (point précédent du même livreur, le même jour, `dans_vehicule = true` ; point courant `false`), 0 sinon. Le premier point de la journée vaut 0.

Affichez les lignes où `changement = 1` pour le livreur `L001` le 2025-01-06.

In [19]:
w_jour = Window.partitionBy("livreur_id", "jour").orderBy("timestamp")
gps = gps.withColumn("changement",
                     F.when(F.lag("dans_vehicule").over(w_jour) & ~F.col("dans_vehicule"), 1).otherwise(0))

l001 = (F.col("livreur_id") == "L001") & (F.col("jour") == "2025-01-06")
gps.where(l001 & (F.col("changement") == 1)).select("timestamp", "livreur_id", "lon", "lat", "changement").show(100)

+-------------------+----------+--------+---------+----------+
|          timestamp|livreur_id|     lon|      lat|changement|
+-------------------+----------+--------+---------+----------+
|2025-01-06 09:54:00|      L001|2.311953|48.871389|         1|
|2025-01-06 10:00:00|      L001|2.313447|48.870848|         1|
|2025-01-06 10:09:00|      L001|2.316012|48.872919|         1|
|2025-01-06 10:23:00|      L001|2.325412|48.874465|         1|
|2025-01-06 10:36:00|      L001| 2.32844|48.866695|         1|
|2025-01-06 10:54:00|      L001|2.337183|48.866435|         1|
|2025-01-06 11:12:00|      L001|2.317523|48.857452|         1|
|2025-01-06 11:38:00|      L001|2.317717|48.850169|         1|
|2025-01-06 12:02:00|      L001|2.312056|48.845401|         1|
|2025-01-06 12:15:00|      L001|2.305215|48.854413|         1|
|2025-01-06 13:04:00|      L001|2.298876|48.845968|         1|
|2025-01-06 13:42:00|      L001|2.326214|48.891988|         1|
|2025-01-06 13:53:00|      L001|2.324882|48.892465|    

5.2) Ajoutez une colonne `numero_livraison` : la somme cumulée de `changement` pour chaque livreur et chaque jour. Elle vaut 0 au chargement du matin.

Pour `L001` le 2025-01-06, affichez par `numero_livraison` le premier et le dernier timestamp, et le nombre de minutes à pied.

In [20]:
w_cumul = w_jour.rowsBetween(Window.unboundedPreceding, Window.currentRow)
gps = gps.withColumn("numero_livraison", F.sum("changement").over(w_cumul).cast("int"))

(gps.where(l001).groupBy("numero_livraison")
 .agg(F.min("timestamp").alias("debut"), F.max("timestamp").alias("fin"),
      nb(~F.col("dans_vehicule")).alias("minutes_a_pied"))
 .orderBy("numero_livraison").show(100))

+----------------+-------------------+-------------------+--------------+
|numero_livraison|              debut|                fin|minutes_a_pied|
+----------------+-------------------+-------------------+--------------+
|               0|2025-01-06 07:43:00|2025-01-06 09:53:00|            44|
|               1|2025-01-06 09:54:00|2025-01-06 09:59:00|             4|
|               2|2025-01-06 10:00:00|2025-01-06 10:08:00|             7|
|               3|2025-01-06 10:09:00|2025-01-06 10:22:00|            10|
|               4|2025-01-06 10:23:00|2025-01-06 10:35:00|             4|
|               5|2025-01-06 10:36:00|2025-01-06 10:53:00|             8|
|               6|2025-01-06 10:54:00|2025-01-06 11:11:00|             4|
|               7|2025-01-06 11:12:00|2025-01-06 11:37:00|            18|
|               8|2025-01-06 11:38:00|2025-01-06 12:01:00|            17|
|               9|2025-01-06 12:02:00|2025-01-06 12:14:00|             4|
|              10|2025-01-06 12:15:00|

5.3) Construisez la table `livraisons` : une ligne par (`livreur_id`, `jour`, `numero_livraison`), calculée sur les seuls points où le livreur est à pied, avec `vehicule_id`, `debut`, `fin`, `duree_min` (nombre de points) et la position moyenne `lon`, `lat`. Combien de lignes contient-elle ?

In [21]:
livraisons = (gps.where(~F.col("dans_vehicule"))
              .groupBy("livreur_id", "jour", "numero_livraison")
              .agg(F.max("vehicule_id").alias("vehicule_id"),
                   F.min("timestamp").alias("debut"),
                   F.max("timestamp").alias("fin"),
                   F.count("*").alias("duree_min"),
                   F.avg("lon").alias("lon"),
                   F.avg("lat").alias("lat"))
              .cache())
print(livraisons.count())

273593


5.4) Ajoutez à chaque livraison ses signatures et ses échecs. Un événement `livree` ou `echec` appartient à une livraison quand il a le même livreur, le même jour, et un horodatage compris entre `debut` et `fin`, avec 1 minute de marge de chaque côté : les points GPS sont à la minute, les événements à la seconde.

Construisez la table `livraisons_ev` : les colonnes de `livraisons`, plus `signature` (horodatage de l'événement), `evenement`, `commande_id` et `client_id`. Une ligne par événement ; une livraison sans événement garde une ligne, avec ces 4 colonnes vides.

In [22]:
evenements = (df_commandes.where(F.col("evenement").isin("livree", "echec"))
              .select("commande_id", "client_id", "livreur_id", F.to_date("horodatage").alias("jour"),
                      F.col("horodatage").alias("signature"), "evenement"))

l, e = livraisons.alias("l"), evenements.alias("e")
minute = F.expr("INTERVAL 1 MINUTE")
condition = ((F.col("l.livreur_id") == F.col("e.livreur_id")) & (F.col("l.jour") == F.col("e.jour"))
             & F.col("e.signature").between(F.col("l.debut") - minute, F.col("l.fin") + minute))

livraisons_ev = l.join(e, condition, "left").select("l.*", "e.signature", "e.evenement", "e.commande_id", "e.client_id")
livraisons_ev.where(F.col("commande_id").isNotNull()).show(5)

+----------+----------+----------------+-----------+-------------------+-------------------+---------+------------------+------------------+-------------------+---------+-----------+---------+
|livreur_id|      jour|numero_livraison|vehicule_id|              debut|                fin|duree_min|               lon|               lat|          signature|evenement|commande_id|client_id|
+----------+----------+----------------+-----------+-------------------+-------------------+---------+------------------+------------------+-------------------+---------+-----------+---------+
|      L002|2025-05-09|               1|        V03|2025-05-09 10:37:00|2025-05-09 10:41:00|        5|2.3160694000000004|        48.8426472|2025-05-09 10:40:08|   livree|   C0072365|  CL01156|
|      L002|2025-05-09|               2|        V03|2025-05-09 10:49:00|2025-05-09 10:57:00|        9| 2.312650777777778|48.842491333333335|2025-05-09 10:55:44|   livree|   C0073549|  CL01912|
|      L002|2025-05-09|            

5.5) Combien d'événements `livree` et `echec` du journal ne sont rattachés à aucune livraison ?

In [27]:
rattaches = (livraisons_ev.where(F.col("commande_id").isNotNull())
             .select("commande_id", "evenement", "signature").distinct())

non_rattaches = evenements.join(rattaches, ["commande_id", "evenement", "signature"], "left_anti")

print("Événements livree/echec du journal :", evenements.count())
print("Non rattachés à une livraison :", non_rattaches.count())
non_rattaches.groupBy("evenement").count().show()

Événements livree/echec du journal : 279902


Non rattachés à une livraison : 0


+---------+-----+
|evenement|count|
+---------+-----+
+---------+-----+



5.6) Toutes les périodes à pied ne sont pas des livraisons. Ajoutez à `livraisons_ev` une colonne `type` :
- `livraison` : au moins un événement ;
- `entrepot` : sinon, position à moins de 300 m d'un entrepôt ;
- `pause` : les autres.

Pour chaque type : nombre de lignes de `livraisons`, durée moyenne, heure moyenne de début.

In [25]:
dist_entrepot = F.least(*[haversine("lon", "lat", F.lit(lo), F.lit(la)) for _, lo, la in ENTREPOTS])

livraisons_ev = (livraisons_ev
                 .withColumn("type", F.when(F.col("commande_id").isNotNull(), "livraison")
                                      .when(dist_entrepot < 0.3, "entrepot")
                                      .otherwise("pause"))
                 .cache())

liv_type = livraisons_ev.select("livreur_id", "jour", "numero_livraison", "debut", "duree_min", "type").distinct().cache()

(liv_type.groupBy("type")
 .agg(F.count("*").alias("nb_livraisons"),
      F.round(F.avg("duree_min"), 1).alias("duree_moyenne_min"),
      F.avg(F.hour("debut") * 60 + F.minute("debut")).alias("m"))
 .select("type", "nb_livraisons", "duree_moyenne_min",
         F.format_string("%02d:%02d", F.floor(F.col("m") / 60), F.floor(F.col("m") % 60)).alias("heure_debut_moyenne"))
 .show())

+---------+-------------+-----------------+-------------------+
|     type|nb_livraisons|duree_moyenne_min|heure_debut_moyenne|
+---------+-------------+-----------------+-------------------+
|    pause|         4967|             37.5|              12:21|
|livraison|       246258|              8.3|              12:11|
| entrepot|        22368|             27.2|              11:40|
+---------+-------------+-----------------+-------------------+



5.7) Une même livraison peut compter plusieurs événements : plusieurs commandes à la même adresse, remises à pied l'une après l'autre. Combien de livraisons ont 1, 2, 3… événements ? Combien concernent au moins 2 clients différents ?

In [28]:
par_livraison = (livraisons_ev.where(F.col("type") == "livraison")
                 .groupBy("livreur_id", "jour", "numero_livraison")
                 .agg(F.count("*").alias("nb_evenements"), F.countDistinct("client_id").alias("nb_clients"))
                 .cache())

par_livraison.groupBy("nb_evenements").count().orderBy("nb_evenements").show()
print("Livraisons avec au moins 2 clients :", par_livraison.where(F.col("nb_clients") >= 2).count())

+-------------+------+
|nb_evenements| count|
+-------------+------+
|            1|217538|
|            2| 24873|
|            3|  3053|
|            4|   592|
|            5|   155|
|            6|    29|
|            7|     8|
|            8|     5|
|            9|     4|
|           10|     1|
+-------------+------+



Livraisons avec au moins 2 clients : 12536


5.8) Ajoutez une colonne `duree_ev_min` à chaque événement : le temps écoulé depuis l'événement précédent de la même livraison, ou depuis le début de la livraison pour le premier. Quelle est la durée moyenne d'une remise signée ? D'un échec ?

In [29]:
w_liv = Window.partitionBy("livreur_id", "jour", "numero_livraison").orderBy("signature")
precedent = F.coalesce(F.lag("signature").over(w_liv), F.col("debut"))
livraisons_ev = livraisons_ev.withColumn(
    "duree_ev_min", (F.col("signature").cast("long") - precedent.cast("long")) / 60)

(livraisons_ev.where(F.col("evenement").isNotNull())
 .groupBy("evenement").agg(F.round(F.avg("duree_ev_min"), 2).alias("duree_moyenne_min")).show())

+---------+-----------------+
|evenement|duree_moyenne_min|
+---------+-----------------+
|    echec|             2.02|
|   livree|             4.24|
+---------+-----------------+



5.9) Pour chaque livreur : nombre de livraisons signées, nombre d'échecs, nombre de jours travaillés, livraisons signées par jour. Affichez les 10 livreurs qui signent le plus de livraisons par jour.

In [30]:
par_livreur = (livraisons.groupBy("livreur_id").agg(F.countDistinct("jour").alias("jours_travailles"))
               .join(livraisons_ev.groupBy("livreur_id")
                     .agg(nb(F.col("evenement") == "livree").alias("signees"),
                          nb(F.col("evenement") == "echec").alias("echecs")),
                     "livreur_id")
               .withColumn("signees_par_jour", F.round(F.col("signees") / F.col("jours_travailles"), 2)))

par_livreur.orderBy(F.desc("signees_par_jour")).show(10)

+----------+----------------+-------+------+----------------+
|livreur_id|jours_travailles|signees|echecs|signees_par_jour|
+----------+----------------+-------+------+----------------+
|      L076|              15|    468|    24|            31.2|
|      L089|              26|    804|    42|           30.92|
|      L070|              18|    543|    28|           30.17|
|      L086|              24|    713|    26|           29.71|
|      L081|              21|    615|    18|           29.29|
|      L072|              17|    490|    21|           28.82|
|      L079|              16|    459|    16|           28.69|
|      L085|              25|    715|    36|            28.6|
|      L080|              16|    457|    18|           28.56|
|      L082|              16|    451|    19|           28.19|
+----------+----------------+-------+------+----------------+
only showing top 10 rows



## 6) Les distances

6.1) Ajoutez à `gps` une colonne `distance_km` : la distance depuis le point précédent du même livreur, le même jour (`haversine`). Le premier point de la journée vaut 0.

*Un point par minute coupe les virages : la distance calculée sous-estime d'environ 12 % la distance réellement parcourue.*

In [35]:
gps = gps.withColumn("distance_km",
                     F.coalesce(haversine(F.lag("lon").over(w_jour), F.lag("lat").over(w_jour), "lon", "lat"),
                                F.lit(0.0))).cache()
gps.count()
gps_base.unpersist()    
gps.select("timestamp", "livreur_id", "changement", "numero_livraison", "distance_km").show()

+-------------------+----------+----------+----------------+--------------------+
|          timestamp|livreur_id|changement|numero_livraison|         distance_km|
+-------------------+----------+----------+----------------+--------------------+
|2025-05-09 07:41:00|      L002|         0|               0|                 0.0|
|2025-05-09 07:42:00|      L002|         0|               0|1.330398417274471E-4|
|2025-05-09 07:43:00|      L002|         0|               0|1.330398417274471E-4|
|2025-05-09 07:44:00|      L002|         0|               0|7.304305398644029E-5|
|2025-05-09 07:45:00|      L002|         0|               0|                 0.0|
|2025-05-09 07:46:00|      L002|         0|               0|7.304305398644029E-5|
|2025-05-09 07:47:00|      L002|         0|               0|                 0.0|
|2025-05-09 07:48:00|      L002|         0|               0|7.304305395400267E-5|
|2025-05-09 07:49:00|      L002|         0|               0|1.330398416022308...|
|2025-05-09 07:5

6.2) Kilomètres parcourus sur l'année par chaque véhicule : les 10 premiers, et le total de la flotte.

In [36]:
km_vehicule = gps.groupBy("vehicule_id").agg(F.round(F.sum("distance_km"), 1).alias("km")).cache()
km_vehicule.orderBy(F.desc("km")).show(10)
km_vehicule.agg(F.sum("km").alias("km_flotte")).show()

+-----------+-------+
|vehicule_id|     km|
+-----------+-------+
|        V36|17442.2|
|        V35|17182.6|
|        V21|17031.3|
|        V20|16900.7|
|        V34|16864.6|
|        V19|16024.8|
|        V41|15516.1|
|        V40|15488.5|
|        V54|15379.7|
|        V53|15358.7|
+-----------+-------+
only showing top 10 rows

+------------------+
|         km_flotte|
+------------------+
|521788.79999999993|
+------------------+



6.3) Kilomètres parcourus sur l'année par chaque livreur : les 10 premiers.

In [37]:
(gps.groupBy("livreur_id").agg(F.round(F.sum("distance_km"), 1).alias("km"))
 .orderBy(F.desc("km")).show(10))

+----------+------+
|livreur_id|    km|
+----------+------+
|      L055|9734.6|
|      L056|9594.9|
|      L064|9511.1|
|      L057|9463.4|
|      L051|9446.3|
|      L050|9364.6|
|      L058|9318.7|
|      L047|9235.2|
|      L059|9225.8|
|      L060|9152.3|
+----------+------+
only showing top 10 rows



6.4) Kilomètres parcourus par jour : moyenne par véhicule-jour, pour chaque entrepôt.

In [38]:
km_jour = gps.groupBy("vehicule_id", "jour").agg(F.sum("distance_km").alias("km"))

(km_jour.join(F.broadcast(vehicule_entrepot), "vehicule_id")
 .groupBy("entrepot").agg(F.round(F.avg("km"), 1).alias("km_moyen_par_vehicule_jour"))
 .orderBy("entrepot").show())

+--------+--------------------------+
|entrepot|km_moyen_par_vehicule_jour|
+--------+--------------------------+
|      E1|                      45.8|
|      E2|                      55.8|
|      E3|                      49.6|
+--------+--------------------------+



6.5) Distance entre deux livraisons : la somme de `distance_km` de chaque (`livreur_id`, `jour`, `numero_livraison`), c'est-à-dire le trajet qui suit la livraison jusqu'à la suivante. Moyenne et médiane, pour les livraisons de type `livraison`.

In [39]:
trajets = gps.groupBy("livreur_id", "jour", "numero_livraison").agg(F.sum("distance_km").alias("km"))

(trajets.join(liv_type.where(F.col("type") == "livraison"), ["livreur_id", "jour", "numero_livraison"])
 .agg(F.round(F.avg("km"), 3).alias("moyenne_km"),
      F.round(F.expr("percentile(km, 0.5)"), 3).alias("mediane_km"))
 .show())

+----------+----------+
|moyenne_km|mediane_km|
+----------+----------+
|     1.566|     0.844|
+----------+----------+



## 7) Les ventes

7.1) Chiffre d'affaires par mois (mois de la commande).

In [40]:
(commandes.groupBy(F.date_format("horodatage", "yyyy-MM").alias("mois"))
 .agg(F.round(F.sum("montant"), 2).alias("chiffre_affaires"))
 .orderBy("mois").show())

+-------+----------------+
|   mois|chiffre_affaires|
+-------+----------------+
|2025-01|      3330814.25|
|2025-02|      3251159.95|
|2025-03|       3775388.3|
|2025-04|      3966928.45|
|2025-05|       4194991.2|
|2025-06|      4410672.05|
|2025-07|       4226598.3|
|2025-08|      3525075.45|
|2025-09|       5566650.7|
|2025-10|       5299971.4|
|2025-11|       6523520.5|
|2025-12|       8250067.4|
+-------+----------------+



7.2) Panier moyen : montant moyen d'une commande.

In [41]:
commandes.agg(F.round(F.avg("montant"), 2).alias("panier_moyen")).show()

+------------+
|panier_moyen|
+------------+
|      207.36|
+------------+



7.3) Panier médian.

In [42]:
commandes.agg(F.expr("percentile(montant, 0.5)").alias("panier_median")).show()

+-------------+
|panier_median|
+-------------+
|        122.8|
+-------------+



7.4) Top 10 des produits par chiffre d'affaires.

In [43]:
(lignes.groupBy("produit_id", "nom").agg(F.round(F.sum("montant"), 2).alias("ca"))
 .orderBy(F.desc("ca")).show(10, truncate=False))

+----------+-----------------------+---------+
|produit_id|nom                    |ca       |
+----------+-----------------------+---------+
|P0096     |Café en grains 6x1 kg  |2247675.0|
|P0049     |Lait demi-écrémé 6x1 L |1621428.8|
|P0097     |Dosettes café x150     |1384866.0|
|P0057     |Comté 12 mois 250 g    |1300460.0|
|P0098     |Gobelets carton x1000  |1141788.0|
|P0018     |Café en grains 1 kg    |1135793.4|
|P0083     |Eau gazeuse 6x1 L      |1129588.2|
|P0150     |Cartouche d'encre noire|1045053.0|
|P0065     |Pavés de saumon x2     |1043281.9|
|P0068     |Oranges 2 kg           |1038729.9|
+----------+-----------------------+---------+
only showing top 10 rows



7.5) Top 10 des produits par quantité vendue.

In [44]:
(lignes.groupBy("produit_id", "nom").agg(F.sum("quantite").alias("quantite_vendue"))
 .orderBy(F.desc("quantite_vendue")).show(10, truncate=False))

+----------+---------------------+---------------+
|produit_id|nom                  |quantite_vendue|
+----------+---------------------+---------------+
|P0082     |Eau minérale 6x1,5 L |290586         |
|P0084     |Eau de source 6x50 cl|289794         |
|P0083     |Eau gazeuse 6x1 L    |289638         |
|P0048     |Lait demi-écrémé 1 L |269998         |
|P0070     |Carottes 1 kg        |268828         |
|P0058     |Mozzarella 125 g     |268634         |
|P0076     |Citrons x4           |268581         |
|P0073     |Salade batavia       |268435         |
|P0053     |Crème fraîche 20 cl  |268346         |
|P0069     |Tomates grappe 1 kg  |268067         |
+----------+---------------------+---------------+
only showing top 10 rows



7.6) Top 10 des clients par dépense sur l'année.

In [45]:
commandes.groupBy("client_id").agg(F.round(F.sum("montant"), 2).alias("depense")).orderBy(F.desc("depense")).show(10)

+---------+---------+
|client_id|  depense|
+---------+---------+
|  CL00034| 436778.7|
|  CL03089| 357505.4|
|  CL02239| 244599.9|
|  CL05203| 212285.5|
|  CL04015| 204201.8|
|  CL03099| 183802.7|
|  CL07784|124598.05|
|  CL03672|112956.85|
|  CL05722|109450.85|
|  CL02517|  92314.6|
+---------+---------+
only showing top 10 rows



7.7) Top 10 des clients par nombre de commandes.

In [46]:
commandes.groupBy("client_id").count().orderBy(F.desc("count")).show(10)

+---------+-----+
|client_id|count|
+---------+-----+
|  CL02239|   64|
|  CL03053|   61|
|  CL04015|   60|
|  CL05203|   59|
|  CL03672|   57|
|  CL07784|   55|
|  CL02517|   54|
|  CL06092|   53|
|  CL05722|   51|
|  CL06965|   50|
+---------+-----+
only showing top 10 rows



## 8) La qualité de service

8.1) Délai entre la commande et sa livraison, en jours calendaires (date de la signature − date de la commande). Combien de commandes livrées pour chaque délai ?

In [47]:
# Une seule passe sur le journal : date de commande et de première signature
cycle = (df_commandes.groupBy("commande_id")
         .agg(F.min(F.when(F.col("evenement") == "commande", F.col("horodatage"))).alias("t_commande"),
              F.min(F.when(F.col("evenement") == "livree", F.col("horodatage"))).alias("t_livree"))
         .withColumn("delai_jours", F.datediff(F.to_date("t_livree"), F.to_date("t_commande")))   # null si non livrée
         .cache())

cycle.where(F.col("delai_jours").isNotNull()).groupBy("delai_jours").count().orderBy("delai_jours").show(50)

+-----------+------+
|delai_jours| count|
+-----------+------+
|          1|198280|
|          2| 53588|
|          3| 12435|
|          4|  3168|
|          5|   923|
|          6|   201|
|          7|    67|
|          8|     9|
|          9|     7|
|         10|     4|
+-----------+------+



8.2) Part des commandes livrées le lendemain, pour chaque mois de commande.

In [48]:
(cycle.groupBy(F.date_format("t_commande", "yyyy-MM").alias("mois_commande"))
 .agg(F.count("*").alias("commandes"),
      F.count("delai_jours").alias("livrees"),
      nb(F.col("delai_jours") == 1).alias("lendemain"))
 .withColumn("part_des_livrees_pct", F.round(100 * F.col("lendemain") / F.col("livrees"), 1))
 .withColumn("part_des_commandes_pct", F.round(100 * F.col("lendemain") / F.col("commandes"), 1))
 .orderBy("mois_commande").show())

+-------------+---------+-------+---------+--------------------+----------------------+
|mois_commande|commandes|livrees|lendemain|part_des_livrees_pct|part_des_commandes_pct|
+-------------+---------+-------+---------+--------------------+----------------------+
|      2025-01|    16186|  16119|    12870|                79.8|                  79.5|
|      2025-02|    15815|  15749|    12573|                79.8|                  79.5|
|      2025-03|    18006|  17936|    13759|                76.7|                  76.4|
|      2025-04|    19008|  18925|    13863|                73.3|                  72.9|
|      2025-05|    20182|  20080|    14093|                70.2|                  69.8|
|      2025-06|    21153|  21049|    15913|                75.6|                  75.2|
|      2025-07|    20803|  20714|    15877|                76.6|                  76.3|
|      2025-08|    16490|  16434|    12408|                75.5|                  75.2|
|      2025-09|    26283|  26182

8.3) Construisez la table `suivi` : une ligne par commande, avec `nb_tentatives` (événements `livree` + `echec`), `nb_echecs` et `issue` (`livree`, `retour` ou `en cours`). Combien de commandes pour chaque nombre de tentatives ?

In [49]:
suivi = (df_commandes.groupBy("commande_id")
         .agg(F.max("client_id").alias("client_id"),
              nb(F.col("evenement").isin("livree", "echec")).alias("nb_tentatives"),
              nb(F.col("evenement") == "echec").alias("nb_echecs"),
              F.max(F.col("evenement") == "livree").alias("livree"),
              F.max(F.col("evenement") == "retour").alias("retour"))
         .withColumn("issue", F.when(F.col("livree"), "livree").when(F.col("retour"), "retour").otherwise("en cours"))
         .drop("livree", "retour")
         .cache())

suivi.groupBy("nb_tentatives").count().orderBy("nb_tentatives").show()

+-------------+------+
|nb_tentatives| count|
+-------------+------+
|            0|  1663|
|            1|260399|
|            2|  9162|
|            3|   393|
+-------------+------+



8.4) Motifs d'échec : nombre d'échecs par motif.

In [50]:
(df_commandes.where(F.col("evenement") == "echec")
 .groupBy(F.col("detail").alias("motif")).count().orderBy(F.desc("count")).show(truncate=False))

+----------------+-----+
|motif           |count|
+----------------+-----+
|absent          |8697 |
|acces_impossible|1372 |
|refus           |1151 |
+----------------+-----+



8.5) Top 10 des clients au plus fort taux d'échec (échecs / tentatives), parmi les clients qui ont au moins 10 tentatives.

In [51]:
(suivi.groupBy("client_id")
 .agg(F.sum("nb_tentatives").alias("tentatives"), F.sum("nb_echecs").alias("echecs"))
 .where(F.col("tentatives") >= 10)
 .withColumn("taux_echec", F.round(F.col("echecs") / F.col("tentatives"), 4))
 .orderBy(F.desc("taux_echec")).show(10))

+---------+----------+------+----------+
|client_id|tentatives|echecs|taux_echec|
+---------+----------+------+----------+
|  CL06349|        78|    39|       0.5|
|  CL05794|        73|    34|    0.4658|
|  CL04500|        69|    32|    0.4638|
|  CL01308|        74|    33|    0.4459|
|  CL02052|        72|    32|    0.4444|
|  CL04097|        67|    29|    0.4328|
|  CL07039|        64|    25|    0.3906|
|  CL00327|        67|    26|    0.3881|
|  CL06732|        14|     4|    0.2857|
|  CL07431|        16|     4|      0.25|
+---------+----------+------+----------+
only showing top 10 rows



## 9) Les clients

On construit la table `clients` colonne par colonne : une ligne par client qui a passé au moins une commande.

9.1) Construisez la table `clients` : `client_id`, `nb_commandes`, `ca` (somme des montants), `panier_moyen`.

9.2) Ajoutez `premiere_commande` et `derniere_commande` (dates).

9.3) Ajoutez `lon` et `lat` : la position moyenne des livraisons où le client a signé. Combien de clients n'ont aucune position ?

9.4) Ajoutez `commune` et `code_postal` : la commune dont le centre (`communes.csv`) est le plus proche de la position du client.

9.5) Les 10 communes qui passent le plus de commandes, avec leur chiffre d'affaires.

9.6) Ajoutez `tentatives`, `echecs` et `taux_echec` (à partir de `suivi`, 8.3).

9.7) Ajoutez `categorie_dominante` (la catégorie où le client dépense le plus) et `part_dominante` (part de sa dépense dans cette catégorie).

9.8) Ajoutez `unites_par_ligne` : la quantité moyenne par ligne de commande.

9.9) Segments de dépense annuelle (`ca`) : moins de 1 000 €, 1 000 à 5 000 €, 5 000 à 20 000 €, 20 000 à 50 000 €, 50 000 € et plus. Ajoutez la colonne `segment`, puis pour chaque segment : nombre de clients, chiffre d'affaires, chiffre d'affaires moyen par client, part du chiffre d'affaires total.

9.10) Nouveaux clients par mois (mois de la première commande), en tableau et en graphique.

9.11) Pour chaque catégorie dominante : nombre de clients et chiffre d'affaires.

9.12) Clients atypiques : les 10 clients dont `unites_par_ligne` est la plus élevée, avec le rapport à la moyenne de tous les clients, leur `ca` et leur `categorie_dominante`.

## 10) Data visualisation

Échelles de couleur : blanc ou clair pour 0, foncé pour les valeurs élevées.

10.1) Heatmap jour de la semaine × heure du nombre de commandes passées.

10.2) Carte du nombre de livraisons signées par commune ou arrondissement (`communes.geojson`, carte choroplèthe).

10.3) Heatmap heure × mois de la vitesse moyenne quand le livreur est à bord et que le véhicule roule.

10.4) Treemap du chiffre d'affaires : catégorie → sous-catégorie.

## 11) Sauvegarde et modélisation

11.2) **Modélisation.** Les analyses précédentes recalculent les mêmes jointures et agrégations. Proposez des collections MongoDB qui rendent ces questions immédiates : pour chaque collection, ce que représente un document, ses champs, ses tableaux imbriqués, et les questions des parties 5 à 9 qu'elle simplifie. Écrivez votre proposition dans la cellule ci-dessous.

*Votre proposition :*